<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> 一书的补充代码，作者 <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 第 6 章：练习题解答

本笔记本使用的软件包：

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

&nbsp;
## 练习 6.1：添加格式感知的奖励塑形

- 我们可以如下所示，使用我们在第 3 章中编写的 `fallback="number_then_full"` 回退策略，在找不到 "\boxed{}" 答案时给予部分奖励（得分 0.5）：

In [ ]:
from reasoning_from_scratch.ch03 import (
    extract_final_candidate, grade_answer
)

def partial_reward_rlvr(answer_text, ground_truth):
    
    # 1) 尝试提取 boxed 答案
    boxed = extract_final_candidate(
        answer_text, fallback=None
    )
    if boxed:
        correct = grade_answer(boxed, ground_truth)
        return 1.0 if correct else 0.0

    # 2) 如果找不到 boxed 答案，则查找数字
    unboxed = extract_final_candidate(
        answer_text, fallback="number_then_full"
    )
    if unboxed:
        correct = grade_answer(unboxed, ground_truth)
        return 0.5 if correct else 0.0

    return 0.0

- 将其插入第 6 章代码并在相同设置下训练时，部分奖励变体的准确率（37.8%）低于标准 GRPO 设置（47.4%），尽管平均使用的 token 数相似

| # | 方法                                     | 步骤 | 最大 token 数 | 轮次数 | 准确率   | 平均 token 数 |
|---|------------------------------------------|------|------------|--------------|----------|----------------|
| 1 | GRPO（第 6 章）                          | 50   | 512        | 8            | 47.4%    | 586.11         |
| 2 | GRPO 部分奖励（练习 6.1）           | 50   | 512        | 8            | 37.8%    | 550.33         |

&nbsp;
## 练习 6.2：零优势情况

- 如果奖励全部相等（例如全为 0 或全为 1），优势将全部为 0，因为减去均值会消除共享的奖励值，只留下零，我们可以在下面演示

In [3]:
import torch

rollout_rewards = [0., 0., 0., 0.]
rewards = torch.tensor(rollout_rewards)
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

print(advantages)

tensor([0., 0., 0., 0.])


In [4]:
rollout_rewards = [1., 1., 1., 1.]
rewards = torch.tensor(rollout_rewards)
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

print(advantages)

tensor([0., 0., 0., 0.])


- 现在，如果所有优势都为 0，损失也将为零，因为损失将优势与对数概率相乘，而乘以零会消除贡献

```python
pg_loss = -(advantages.detach() * logps).mean()
```

- 因此，策略梯度为零，模型参数不会针对该提示进行更新

- 这种行为是有意为之的；如果所有轮次同样差或同样好，就没有相对信号来告诉模型应该强化或抑制哪种行为
- 直觉上，如果模型正确回答了所有问题，就不需要更新它
- 反之，如果模型回答错了所有问题，我们不希望更新模型来强化这种行为